In [32]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output


def add_white_padding(img, output_path, padding=100):
    # 2. Add the border using copyMakeBorder
    # BORDER_CONSTANT tells OpenCV to fill the border with a solid color
    # We define the color as white (255, 255, 255) for BGR format
    padded_img = cv2.copyMakeBorder(
        img, 
        padding,       # Top padding
        padding,       # Bottom padding
        padding,       # Left padding
        padding,       # Right padding
        cv2.BORDER_CONSTANT, 
        value=[255, 255, 255]  # White color
    )

    # 3. Save the result
    cv2.imwrite(output_path, padded_img)
    print(f"Image saved with {padding}px white padding on all sides: {output_path}")

    return padded_img

#======================= sort_contours ===========================================
def sort_contours(cnts, method="left-to-right"):
        # initialize the reverse flag and sort index
        reverse = False
        i = 0

        # handle if we need to sort in reverse
        if method == "right-to-left" or method == "bottom-to-top":
                reverse = True

        # handle if we are sorting against the y-coordinate rather than
        # the x-coordinate of the bounding box
        if method == "top-to-bottom" or method == "bottom-to-top":
                i = 1

        # construct the list of bounding boxes and sort them from top to
        # bottom
        boundingBoxes = [cv2.boundingRect(c) for c in cnts]
        (cnts, boundingBoxes) = zip(*sorted(zip(cnts, boundingBoxes),
                key=lambda b:b[1][i], reverse=reverse))

        # return the list of sorted contours and bounding boxes
        return (cnts, boundingBoxes)
#==================================================================================


def box_extraction(img_for_box_extraction_path, cropped_dir_path, h_val, v_val):
        img = cv2.imread(img_for_box_extraction_path, 0)# Read the image
        (thresh, img_bin) = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)#Thresholding the image
        
        # img_bin = add_white_padding(img_bin, "inter_processing/Image_bin_with_padding.jpg")
        
        img_bin = 255-img_bin# Invert the image
        cv2.imwrite("inter_processing/Image_bin.jpg",img_bin)

        # Defining a kernel length
        # kernel_length_horizont = np.array(img).shape[1]//90
        kernel_length_horizont = h_val
        kernel_length_vertical = v_val

        print(f'kernel_length_horizont: {kernel_length_horizont}')
        print(f'kernel_length_vertical: {kernel_length_vertical}')

        # A verticle kernel of (1 X kernel_length), which will detect all the verticle lines from the image.
        verticle_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_length_vertical))

        # A horizontal kernel of (kernel_length X 1), which will help to detect all the horizontal line from the image.
        hori_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_length_horizont, 1))

        # A kernel of (3 X 3) ones.
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

        # # Morphological operation to detect verticle lines from an image
        # img_temp1 = cv2.dilate(img_bin, verticle_kernel, iterations=3)
        # verticle_lines_img = cv2.erode(img_temp1, verticle_kernel, iterations=3)
        # cv2.imwrite("inter_processing/verticle_lines.jpg",verticle_lines_img)

        # # Morphological operation to detect horizontal lines from an image
        # img_temp2 = cv2.dilate(img_bin, hori_kernel, iterations=3)
        # horizontal_lines_img = cv2.erode(img_temp2, hori_kernel, iterations=3)
        # cv2.imwrite("inter_processing/horizontal_lines.jpg",horizontal_lines_img)


        # Morphological operation to detect verticle lines from an image
        img_temp1 = cv2.erode(img_bin, verticle_kernel, iterations=3)
        verticle_lines_img = cv2.dilate(img_temp1, verticle_kernel, iterations=3)
        cv2.imwrite("inter_processing/verticle_lines.jpg",verticle_lines_img)

        # Morphological operation to detect horizontal lines from an image
        img_temp2 = cv2.erode(img_bin, hori_kernel, iterations=3)
        horizontal_lines_img = cv2.dilate(img_temp2, hori_kernel, iterations=3)
        cv2.imwrite("inter_processing/horizontal_lines.jpg",horizontal_lines_img)



        # Weighting parameters, this will decide the quantity of an image to be added to make a new image.
        alpha = 0.5
        beta = 1.0 - alpha

        # This function helps to add two image with specific weight parameter to get a third image a s summation of two image.
        img_final_bin = cv2.addWeighted(verticle_lines_img, alpha, horizontal_lines_img, beta, 0.0)
        img_final_bin = cv2.erode(~img_final_bin, kernel, iterations=2)
        (thresh, img_final_bin) = cv2.threshold(img_final_bin, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

        # For Debugging
        # Enable this line to see verticle and horizontal lines in the image which is used to find boxes
        cv2.imwrite("inter_processing/img_final_bin.jpg",img_final_bin)

        # Find contours for image, which will detect all the boxes
        contours, hierarchy = cv2.findContours(img_final_bin, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

        # Sort all the contours by top to bottom.
        (contours, boundingBoxes) = sort_contours(contours, method="top-to-bottom")
        idx = 0
        for c in contours:
                # Returns the location and width,height for every contour
                x, y, w, h = cv2.boundingRect(c)

                # If the box height is greater then 20, widht is >80, then only save it as a box in "cropped/" folder.
                # if (w > 80 and h > 20) and w > 3*h:
                #         idx += 1
                #         new_img = img[y:y+h, x:x+w]
                #         cv2.imwrite(cropped_dir_path+str(idx) + '.png', new_img)


def update_image(change):
        print('update_image')
        
        h_val = slider_h.value
        v_val = slider_v.value
        # box_extraction("/media/vadim/1TB_SSD/my_github/computer-vision-document-table-parser/input_images/1_f.jpg", "./output/", h_val, v_val)
        box_extraction("/media/vadim/1TB_SSD/my_github/computer-vision-document-table-parser/input_images/1_rotate.jpg", "./output/", h_val, v_val)


Path("./output/").mkdir(parents=True, exist_ok=True)
Path("./inter_processing/").mkdir(parents=True, exist_ok=True)

# Create widgets
slider_h = widgets.IntSlider(value=10, min=1, max=100, step=1, description='Horiz Kernel:')
slider_v = widgets.IntSlider(value=8, min=1, max=100, step=1, description='Vert Kernel:')

# ==================== BIND EVENTS & SHOW ====================
# Connect sliders to the update function
slider_h.observe(update_image, names='value')
slider_v.observe(update_image, names='value')

# Display widgets and the initial image
display(widgets.VBox([slider_h, slider_v]))


# update_image(None)




In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==================== YOUR HELPER FUNCTIONS ====================
def add_white_padding(img, output_path, padding=100):
    padded_img = cv2.copyMakeBorder(
        img, padding, padding, padding, padding,
        cv2.BORDER_CONSTANT, value=[255, 255, 255]
    )
    cv2.imwrite(output_path, padded_img)
    return padded_img

def sort_contours(cnts, method="left-to-right"):
    reverse = False
    i = 0
    if method == "right-to-left" or method == "bottom-to-top":
        reverse = True
    if method == "top-to-bottom" or method == "bottom-to-top":
        i = 1
    boundingBoxes = [cv2.boundingRect(c) for c in cnts]
    (cnts, boundingBoxes) = zip(*sorted(zip(cnts, boundingBoxes),
            key=lambda b:b[1][i], reverse=reverse))
    return (cnts, boundingBoxes)

# ==================== YOUR CORE FUNCTION (MODIFIED FOR PLOTTING) ====================
def process_box_extraction(img_path, h_kernel, v_kernel, cropped_dir_path="./output/"):
    # 1. Read image
    img = cv2.imread(img_path, 0)
    (thresh, img_bin) = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    img_bin = 255 - img_bin # Invert

    # 2. Define kernels
    verticle_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, v_kernel))
    hori_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (h_kernel, 1))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

    # 3. Morph ops for lines
    img_temp1 = cv2.erode(img_bin, verticle_kernel, iterations=3)
    verticle_lines_img = cv2.dilate(img_temp1, verticle_kernel, iterations=3)

    img_temp2 = cv2.erode(img_bin, hori_kernel, iterations=3)
    horizontal_lines_img = cv2.dilate(img_temp2, hori_kernel, iterations=3)

    # 4. Combine lines and find boxes
    alpha = 0.5
    beta = 1.0 - alpha
    img_final_bin = cv2.addWeighted(verticle_lines_img, alpha, horizontal_lines_img, beta, 0.0)
    img_final_bin = cv2.erode(~img_final_bin, kernel, iterations=2)
    (thresh, img_final_bin) = cv2.threshold(img_final_bin, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    # 5. Find Contours
    contours, hierarchy = cv2.findContours(img_final_bin, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    (contours, boundingBoxes) = sort_contours(contours, method="top-to-bottom")

    # 6. Prepare visualization image (Color copy to draw rectangles)
    vis_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    
    idx = 0
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        # Filter boxes
        if (w > 80 and h > 20) and w > 3*h:
            idx += 1
            # Draw green rectangle
            cv2.rectangle(vis_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
            # Uncomment the next line if you want to actually save cropped images while sliding
            # new_img = img[y:y+h, x:x+w]
            # cv2.imwrite(f"{cropped_dir_path}{idx}.png", new_img)

    # Add text info on image
    text = f"Horizontal Kernel: {h_kernel} | Vertical Kernel: {v_kernel} | Boxes: {idx}"
    cv2.putText(vis_img, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Convert BGR (OpenCV format) to RGB (Matplotlib format) for display
    return cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)

# ==================== SETUP RUNTIME WIDGETS IN JUPYTER ====================
# Path to your image
INPUT_IMAGE = "/media/vadim/1TB_SSD/my_github/computer-vision-document-table-parser/input_images/1_rotate.jpg" 

# Create widgets
slider_h = widgets.IntSlider(value=15, min=3, max=100, step=2, description='Horiz Kernel:')
slider_v = widgets.IntSlider(value=7, min=3, max=100, step=2, description='Vert Kernel:')
output_ui = widgets.Output()

# Create directories
Path("./output/").mkdir(parents=True, exist_ok=True)
Path("./inter_processing/").mkdir(parents=True, exist_ok=True)

# ==================== UPDATE FUNCTION ====================
def update_image(change):
    with output_ui:
        clear_output(wait=True) # Clear previous image
        # Get values from sliders
        h_val = slider_h.value
        v_val = slider_v.value
        
        # Process
        result_img = process_box_extraction(INPUT_IMAGE, h_val, v_val)
        
        # Display using Matplotlib inline
        plt.figure(figsize=(15, 10)) # Adjust figure size as needed
        plt.imshow(result_img)
        plt.axis('off')
        plt.show()

# ==================== BIND EVENTS & SHOW ====================
# Connect sliders to the update function
slider_h.observe(update_image, names='value')
slider_v.observe(update_image, names='value')

# Display widgets and the initial image
display(widgets.VBox([slider_h, slider_v, output_ui]))
update_image(None) # Trigger initial render